# 2 · Chroma Database Builder

Loads the Markdown transcriptions from **`output_transcription_md/`** into a Chroma vector database
(**`chroma-db/`**) for the question-answering notebook.

- The list of files to load comes from **`output_transcr_manifest.csv`**; other files in the folder are ignored.
- Each file's metadata (`document_type`, `sender`, `recipient`, `place`, `date`, `title`, `needs_review`, …)
  is stored with it, so you can filter by it in notebook 3. Two fields are added for filtering by date:
  `date_num` (e.g. `18670314`) and `year`.
- The transcription and the marginalia are searchable; the `## For review` notes are not added to the searchable text.
- Normally only **new or changed** files are embedded. Files that are no longer in the manifest are removed.
- The embedding provider and model come from `pipeline.env`. If you change them, the database is rebuilt
  automatically, because vectors from different models cannot be mixed.

The code is in `chroma_builder.py`.

## Setup

All settings shared by the three notebooks (API keys, Claude model, embedding model, folder names) live in
**`pipeline.env`** in this folder. Open it, fill in your keys, choose the model, save, then run this cell.
If you change `pipeline.env` later, just run this cell again.

In [1]:
from pipeline_common import load_settings
import chroma_builder

settings = load_settings()

Claude model      : claude-opus-5-5  (effort: high)
Embeddings        : voyage / voyage-3.5
ANTHROPIC_API_KEY : set (...ugAA)
VOYAGE_API_KEY    : set (...Frs4)
Project folder    : /Users/rfreedma/Documents/CRIM_Python/RAG CHROMA MP


## Run options

**`FORCE_REBUILD`**: `False` (normal) adds only new or changed transcriptions.
`True` empties the database and embeds every transcription again.

In [2]:
FORCE_REBUILD = False

## Step 1 · Preview

Shows each transcription with its status (`new`, `changed`, `unchanged`), number of chunks, and the
metadata that will be stored. No API calls are made.

In [3]:
chroma_builder.preview(settings)

The database is empty or was built with other embedding settings: every file will be loaded.

Transcriptions in manifest: 27  ->  {'new': 27}


,status,chunks,imagen_fuente,recorte,diario,fecha,fecha_fuente,titular,idioma,revision,coleccion,modelo,procesado,fecha_num,anio,source_file,anotacion
0,new,2,IMG_1950.JPEG,1,El Pueblo,1940-07-13,manuscrita,Ofreciéronse anoche novedades en el Colón y Odeón,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400713,1940,IMG_1950_r01.md,"""El Pueblo"" Julio 13 de 1940."
1,new,2,IMG_1950.JPEG,2,El Pueblo,1940-07-12,manuscrita,"Se trata del baile ""Panambi"" de A. E. Ginastera",Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400712,1940,IMG_1950_r02.md,"""El Pueblo"" Julio 12 de 1940."
2,new,2,IMG_1951.JPEG,1,Rumbos,1940-09-28,manuscrita,Alberto E. Ginastera,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400928,1940,IMG_1951_r01.md,"""RUMBOS"" Septiembre 28 de 1940."
3,new,1,IMG_1951.JPEG,2,Rumbos,1940-08-17,manuscrita,"El autor del ballet ""Panambi"" es un joven veci...",Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400817,1940,IMG_1951_r02.md,"""RUMBOS"" Agosto 17 de 1940."
4,new,3,IMG_1952.JPEG,1,La Libertad,1940-07-13,manuscrita,"Fué estrenado anoche el ballet ""Panambí"" en el...",Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400713,1940,IMG_1952_r01.md,"""""LA LIBERTAD"" (AVELLANEDA) JULIO 13 DE 1940."""
5,new,1,IMG_1953.JPEG,1,El Diario,1940-07-21,manuscrita,El jurado adjudico los premios municipales a l...,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400721,1940,IMG_1953_r01.md,"""El Diario"" Julio 21 de 1940."
6,new,1,IMG_1953.JPEG,2,La Nación,1940-07-20,manuscrita,Fueron adjudicados ayer los premios municipale...,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400720,1940,IMG_1953_r02.md,"""La Nación"" Julio 20 de 1940."
7,new,1,IMG_1953.JPEG,3,Ahora,1940-08-27,manuscrita,Entrego el intendente los premios musicales de...,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400827,1940,IMG_1953_r03.md,"""Ahora"" Agosto 27 de 1940."
8,new,1,IMG_1953.JPEG,4,La Prensa,1940-07-20,manuscrita,Fueron Otorgados los Premios del Concurso De M...,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400720,1940,IMG_1953_r04.md,"""La Prensa"" Julio 20 de 1940."
9,new,1,IMG_1954.JPEG,1,Crítica,1940-07-21,manuscrita,Premios municipales de musica,Spanish,True,"Ginastera scrapbook, press clippings 1940 (pho...",claude-opus-5-5,2026-10-03,19400721,1940,IMG_1954_r01.md,"""“CRÍTICA” JULIO 21 DE 1940."""


## Step 2 · Build  (uses the embeddings API)

In [4]:
chroma_builder.build(settings, force=FORCE_REBUILD)

Embedding settings differ from the ones this database was built with:
   database : {'version': '3.0', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'transcriptions', 'source_path': '/Users/rfreedma/Downloads/Sources/Input_MD'}
   now      : {'version': '4.0', 'embedding_provider': 'voyage', 'embedding_model': 'voyage-3.5', 'collection_name': 'transcriptions'}
The database will be rebuilt from scratch.


/Users/rfreedma/Documents/CRIM_Python/RAG CHROMA MP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  added   IMG_1950_r01.md (2 chunk(s))
  added   IMG_1950_r02.md (2 chunk(s))
  added   IMG_1951_r01.md (2 chunk(s))
  added   IMG_1951_r02.md (1 chunk(s))
  added   IMG_1952_r01.md (3 chunk(s))
  added   IMG_1953_r01.md (1 chunk(s))
  added   IMG_1953_r02.md (1 chunk(s))
  added   IMG_1953_r03.md (1 chunk(s))
  added   IMG_1953_r04.md (1 chunk(s))
  added   IMG_1954_r01.md (1 chunk(s))
  added   IMG_1954_r02.md (1 chunk(s))
  added   IMG_1954_r03.md (1 chunk(s))
  added   IMG_1955_r01.md (1 chunk(s))
  added   IMG_1955_r02.md (3 chunk(s))
  added   IMG_1955_r03.md (2 chunk(s))
  added   IMG_1956_r01.md (1 chunk(s))
  added   IMG_1956_r02.md (1 chunk(s))
  added   IMG_1956_r03.md (1 chunk(s))
  added   IMG_1956_r04.md (1 chunk(s))
  added   IMG_1957_r01.md (1 chunk(s))
  added   IMG_1957_r02.md (2 chunk(s))
  added   IMG_1957_r03.md (4 chunk(s))
  added   IMG_1957_r04.md (1 chunk(s))
  added   IMG_1958_r01.md (1 chunk(s))
  added   IMG_1959_r01.md (3 chunk(s))
  added   IMG_1959_r02.md

## Step 3 · Check the database

A summary, then a quick test search (no Claude call; this only checks that retrieval works).

In [5]:
chroma_builder.summary(settings)

Chunks in database : 41
Transcriptions     : 27
Newspapers         : Ahora, Crítica, El Diario, El Pueblo, La Acción, La Capital, La Hora, La Libertad, La Nación, La Prensa, La Razón, La Verdad, Rumbos, Símbolo
Dates              : 1940-07-12 to 1940-11-25
Metadata fields    : anio, anotacion, chunk, chunks_total, coleccion, diario, fecha, fecha_fuente, fecha_num, file_hash, idioma, imagen_fuente, modelo, procesado, recorte, revision, source_file, titular


In [ ]:
chroma_builder.test_search(settings, "Brief von Liszt an Gottschalg", k=3)